# GCD-11 with Facebook as the reference graph

This notebook explores **Graphlet Correlation Distance (GCD-11)** using Facebook as the reference graph. This metric is different from the others: it compares **two graphs**, rather than summarizing a single graph with one scalar.

## Metric definition

For a graph $G$, let $X_G$ be the node-by-orbit matrix for the 11 non-redundant graphlet orbits. Form the graphlet correlation matrix

$$\mathrm{GCM}_{ab}(G)=\rho_s(X_G[:,a], X_G[:,b]),$$

where $\rho_s$ is Spearman correlation. Then

$$\mathrm{GCD}_{11}(G,H)=\left\|\operatorname{vec}_{\triangle}(\mathrm{GCM}(G)) - \operatorname{vec}_{\triangle}(\mathrm{GCM}(H))\right\|_2.$$

Interpretation: summarize how graphlet roles co-vary across nodes in each graph, then measure how different those summaries are. Lower is more similar.

In [ ]:

from pathlib import Path
import sys
import math
import itertools
import shutil

import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / 'metrics.py').exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR.parent

sys.path.insert(0, str(NOTEBOOK_DIR))
from metrics import (
    load_graph,
    unique_triangle_count,
    wedge_count,
    gcc,
    alcc,
    count_k_cliques,
    k_clique_density,
    higher_order_global_clustering,
    higher_order_average_local_clustering,
    higher_order_local_clustering,
    node_k_clique_membership_counts,
    gcd11,
    orca_node_orbits_4,
    gcm11_from_orbits,
    GCD11_ORBITS,
)

DATA_PATH = NOTEBOOK_DIR.parent / 'data' / 'gt_txt' / 'facebook.txt'
G = load_graph(DATA_PATH)
print(f'Loaded {DATA_PATH.name}: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')


In [ ]:
repo_root = NOTEBOOK_DIR.parent
facebook_path = repo_root / 'data' / 'gt_txt' / 'facebook.txt'
hamster_path = repo_root / 'data' / 'gt_txt' / 'hamsterster.txt'
polblogs_path = repo_root / 'data' / 'gt_txt' / 'polblogs.txt'
G_fb = load_graph(facebook_path)
G_ham = load_graph(hamster_path)
G_pol = load_graph(polblogs_path)

orca_path = shutil.which('orca')
print('ORCA found at:' if orca_path else 'ORCA not found.', orca_path)

## If ORCA is available, compute GCD-11

The cell below compares Facebook to itself, to hamsterster, and to polblogs. Facebook-to-itself should be 0 (or numerically extremely close).

In [ ]:
if orca_path is None:
    print('Install ORCA and ensure it is on PATH to run this section.')
else:
    rows = []
    for name, H in [('facebook', G_fb), ('hamsterster', G_ham), ('polblogs', G_pol)]:
        rows.append({'reference': 'facebook', 'other_graph': name, 'gcd11': gcd11(G_fb, H, orca_path=orca_path)})
    pd.DataFrame(rows)

## Inspect the graphlet correlation matrix itself

The matrix is often as educational as the distance. It shows which graphlet roles tend to rise and fall together across nodes.

In [ ]:
if orca_path is None:
    print('Install ORCA and ensure it is on PATH to run this section.')
else:
    O_fb = orca_node_orbits_4(G_fb, orca_path=orca_path)
    GCM_fb = gcm11_from_orbits(O_fb)
    plt.figure(figsize=(6,5))
    plt.imshow(GCM_fb, vmin=-1, vmax=1)
    plt.colorbar(label='Spearman correlation')
    plt.title('Facebook GCM (11 non-redundant orbits)')
    plt.xlabel('orbit index in GCD-11 basis')
    plt.ylabel('orbit index in GCD-11 basis')
    plt.show()

## Interpretation

GCD-11 is the broadest realism metric in this bundle.

For realism testing later:
- **small GCD-11** to Facebook means the generated graph is locally topologically similar to Facebook
- **large GCD-11** means it is structurally different, even if it happens to match one or two scalar metrics like GCC or ALCC